[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/06-dsa-interview/04_stack-queue.ipynb)

*DSA Interview Handbook · Notebook 4 of 14*

# Stack & Queue

Last-in-first-out for matching and 'next greater' questions; first-in-first-out for order.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A stack is a pile of plates and a queue is a checkout line. A monotonic stack remembers only what can still matter.

**What interviewers really test here:** Matching brackets, monotonic stacks for 'next greater' and amortised cost.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. The stack: last in, first out *(Basics)*
- [ ] 2. The queue: first in, first out *(Basics)*
- [ ] 3. Matching and nesting *(Core)*
- [ ] 4. Evaluating expressions *(Core)*
- [ ] 5. Monotonic stack: next greater element *(Core)*
- [ ] 6. Monotonic stack for ranges and contributions *(Advanced)*
- [ ] 7. Designing with stacks and queues *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

A **stack** is a linear **abstract data type (ADT)** that follows the **LIFO** principle — Last In, First Out. Elements are added (**push**) and removed (**pop**) only at one end, called the **top**.



A **queue** is a linear ADT that follows the **FIFO** principle — First In, First Out. Elements are added at the **rear** (**enqueue**) and removed from the **front** (**dequeue**).



An ADT describes *what* operations exist, not *how* they are stored. Both can be implemented with an array or a linked list.

### Stack operations

| Operation | Meaning | Python (`list`) | Time |
| --- | --- | --- | --- |
| push(x) | add x on top | `st.append(x)` | O(1) amortised |
| pop() | remove and return the top | `st.pop()` | O(1) |
| peek() / top() | read the top | `st[-1]` | O(1) |
| isEmpty() | no elements? | `not st` | O(1) |
| size() | number of elements | `len(st)` | O(1) |



**Stack overflow**: pushing onto a full fixed-size stack (or recursion too deep). **Stack underflow**: popping from an empty stack.

### Queue operations and types

| Operation | Python (`collections.deque`) | Time |
| --- | --- | --- |
| enqueue(x) | `q.append(x)` | O(1) |
| dequeue() | `q.popleft()` | O(1) |
| front() | `q[0]` | O(1) |
| rear() | `q[-1]` | O(1) |



#### Types of queue



| Type | Description |
| --- | --- |
| **Simple queue** | insert at rear, delete at front |
| **Circular queue** | a fixed array where the rear wraps around to index 0, reusing freed slots: `rear = (rear + 1) % capacity` |
| **Deque** (double-ended) | insert and delete at both ends in O(1) |
| **Priority queue** | removes the highest-priority element first (implemented with a heap — see the Heap topic) |
| **Monotonic queue / stack** | kept in sorted order by popping elements that break it; used for window max and next-greater problems |

### Implementations

| Implementation | Stack | Queue |
| --- | --- | --- |
| Dynamic array | push/pop at the end: O(1) | dequeue at index 0 shifts everything: O(n) — use a circular buffer instead |
| Circular array | — | front and rear indices wrap with `% capacity`: O(1), fixed size |
| Linked list | push/pop at the head: O(1) | enqueue at tail, dequeue at head: O(1) |
| Two stacks | — | amortised O(1) (pour 'in' into 'out' when 'out' is empty) |



In Python use `list` for stacks and `collections.deque` for queues. `queue.Queue` is for thread-safe producer/consumer code and is slower.

### Advantages and disadvantages

**👍 Pros**

##### Advantages

- All core operations O(1)
- Natural model for nesting (stack) and fairness/order (queue)
- Simple to implement and reason about



**👎 Cons**

##### Disadvantages

- No random access or search (O(n))
- Fixed-size array versions can overflow
- Only the ends are accessible

### Applications

##### Stack

- Function call stack and recursion
- Undo / redo in editors, browser back button
- Balanced parentheses, HTML/XML tag matching
- Expression evaluation and infix → postfix conversion
- Depth-first search, backtracking
- Monotonic stack: stock span, histogram area



##### Queue

- CPU and printer scheduling, task queues
- Breadth-first search
- Buffers: keyboard input, IO, streaming
- Message queues between services (Kafka, RabbitMQ)
- Handling requests in arrival order (web servers, call centres)

### Expression notations

| Notation | Example for (2 + 3) × 4 | Note |
| --- | --- | --- |
| Infix | `(2 + 3) * 4` | operator between operands; needs brackets and precedence |
| Prefix (Polish) | `* + 2 3 4` | operator first; no brackets needed |
| Postfix (Reverse Polish) | `2 3 + 4 *` | operator last; evaluated left to right with one stack |



The **shunting-yard algorithm** converts infix to postfix with an operator stack, respecting precedence and brackets.

---
# Part 2: Lessons, step by step

## Lesson 1: The stack: last in, first out

*Level: Basics*

A **stack** is like a pile of plates: you only add to the top and take from the top. The last plate put on is the first one taken off — **LIFO**.



| Action | Python list | Cost |
| --- | --- | --- |
| push x | `st.append(x)` | O(1) |
| pop the top | `st.pop()` | O(1) |
| look at the top | `st[-1]` | O(1) |
| is it empty? | `not st` | O(1) |



You already use one every day: the **call stack**. When a function calls another, Python pushes a frame; when it returns, the frame is popped. That's why recursion and stacks are interchangeable.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def stack_lesson():
    st = []
    for op, x in [("push", 3), ("push", 8), ("push", 5), ("pop", None), ("push", 1), ("pop", None), ("pop", None)]:
        if op == "push":
            st.append(x)
        else:
            y = st.pop()
    return st


stack_lesson()

[3]

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. push(3) — goes on top
2. push(8) — goes on top
3. push(5) — goes on top
4. pop() → 5, the most recent item
5. push(1) — goes on top
6. pop() → 1, the most recent item
7. pop() → 8, the most recent item

</details>


**✏️ Your turn:** you now have what you need for [Q5](#q5). Try one now, or carry on and do them in Part 3.

## Lesson 2: The queue: first in, first out

*Level: Basics*

A **queue** is a line at a ticket counter: join at the back, leave from the front — **FIFO**. Use `collections.deque`:



```python
from collections import deque
q = deque()
q.append(x)      # join at the back   O(1)
q.popleft()      # leave the front    O(1)
q[0]             # peek the front     O(1)
```



**⚠️ Watch out:** Don't use `list.pop(0)` as a queue — it shifts every element, O(n) per pop.



Queues are the heart of **BFS** (graphs and trees), task scheduling and anything processed "in arrival order".

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def queue_lesson():
    from collections import deque
    q = deque()
    for op, x in [("in", "A"), ("in", "B"), ("in", "C"), ("out", None), ("in", "D"), ("out", None)]:
        if op == "in":
            q.append(x)
        else:
            y = q.popleft()
    return list(q)


queue_lesson()

['C', 'D']

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. append('A') — joins at the back
2. append('B') — joins at the back
3. append('C') — joins at the back
4. popleft() → 'A', who arrived first
5. append('D') — joins at the back
6. popleft() → 'B', who arrived first

</details>


**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q4](#q4). Try one now, or carry on and do them in Part 3.

## Lesson 3: Matching and nesting

*Level: Core*

Whenever things **open and close** — brackets, tags, folders, nested encodings — the most recently opened thing must close first. That's exactly LIFO.



```python
pair = {')': '(', ']': '[', '}': '{'}
st = []
for ch in s:
    if ch in pair:                       # a closer
        if not st or st.pop() != pair[ch]:
            return False
    else:
        st.append(ch)                    # an opener
return not st                            # nothing left open
```



Variations: **decode string** pushes (text so far, repeat count) at every `[`; **simplify path** pushes folder names and pops on `..`; **remove adjacent duplicates** uses the stack as the answer itself.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q12](#q12), [Q13](#q13), [Q11](#q11), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 4: Evaluating expressions

*Level: Core*

Computers evaluate expressions with stacks.




- **Reverse Polish notation** (`2 1 + 3 *`): numbers are pushed; an operator pops two numbers, applies itself, and pushes the result.
- **Infix with + − \* /** (no brackets): \* and / bind tighter, so do them immediately with the stack top; push +x or −x and sum the stack at the end.
- With brackets, push the current result and sign at `(`, restore at `)`.



**⚠️ Watch out:** Python's `//` rounds toward −∞. Problems usually want rounding toward 0: use `int(a / b)`.

**✏️ Your turn:** you now have what you need for [Q7](#q7), [Q17](#q17). Try one now, or carry on and do them in Part 3.

## Lesson 5: Monotonic stack: next greater element

*Level: Core*

"For each element, find the next element to the right that is greater" is O(n²) with two loops. A **monotonic stack** does it in O(n).



Keep a stack of indices whose values are **decreasing**. These are elements still waiting for a bigger value. When a new value arrives:




1. While it is bigger than the value on top, pop the top — the new value is its answer.
2. Push the new index; it now waits too.



Each index is pushed and popped once → O(n). Flip the comparison for "next smaller"; scan from the right for "previous greater"; loop twice for circular arrays.

**✏️ Your turn:** you now have what you need for [Q8](#q8), [Q2](#q2), [Q9](#q9), [Q15](#q15). Try one now, or carry on and do them in Part 3.

## Lesson 6: Monotonic stack for ranges and contributions

*Level: Advanced*

A monotonic stack can tell you, for every element, how far it extends left and right before hitting something smaller. That solves "range" problems:




- **Largest rectangle in histogram**: when a bar is popped, the bar that popped it is its right limit and the new stack top is its left limit → width known, area known.
- **Sum of subarray minimums**: `arr[i]` is the minimum of `left[i] × right[i]` subarrays, so add `arr[i] × left[i] × right[i]`. Use `<` on one side and `<=` on the other so equal values aren't counted twice.
- **Remove K digits**: pop bigger digits while removals remain — a greedy that keeps the stack increasing.
- **Maximal rectangle**: run the histogram solution on every row.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def histogram_lesson():
    heights = [2, 1, 5, 6, 2, 3]
    stack, best = [], 0
    for i in range(len(heights) + 1):
        h = heights[i] if i < len(heights) else 0
        while stack and heights[stack[-1]] >= h:
            top = stack.pop()
            left = stack[-1] + 1 if stack else 0
            best = max(best, heights[top] * (i - left))
        stack.append(i)
    return best


histogram_lesson()

10

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Pop bar 2: it stretches from 0 to 0 → area 2
2. Pop bar 6: it stretches from 3 to 3 → area 6
3. Pop bar 5: it stretches from 2 to 3 → area 10
4. Pop bar 3: it stretches from 5 to 5 → area 3
5. Pop bar 2: it stretches from 2 to 5 → area 8
6. Pop bar 1: it stretches from 0 to 5 → area 6

</details>


**✏️ Your turn:** you now have what you need for [Q18](#q18), [Q16](#q16), [Q14](#q14), [Q19](#q19), [Q10](#q10). Try one now, or carry on and do them in Part 3.

## Lesson 7: Designing with stacks and queues

*Level: Advanced*

Design questions ask you to build a structure with given costs. Two tricks cover most of them:




- **Store extra info with each item.** Min Stack pushes `(value, min so far)`; Stock Span pushes `(price, span)`.
- **Amortised analysis.** A queue from two stacks pours `in` into `out` only when `out` is empty. A single pop can cost O(n), but each element is moved at most once in its lifetime, so the **average** cost per operation is O(1).



In an interview, state the cost of every method and say "amortised" when it applies.

**✏️ Your turn:** you now have what you need for [Q6](#q6), [Q3](#q3), [Q4](#q4), [Q15](#q15). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Valid Parentheses

🟢 Warm-up · *Matching stack*

Given a string of `()[]{}`, return `True` if every bracket is closed by the same type in the right order.

**Examples**

- `s = '()[]{}'` → `True`
- `s = '(]'` → `False`
- `s = '([])'` → `True`

In [ ]:
def is_valid(s):
    """
    Given a string of ()[]{}, return True if every bracket is closed by the same
    type in the right order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_valid, [
    (('()[]{}',), True),
    (('(]',), False),
    (('([])',), True),
    (('([)]',), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeatedly delete adjacent pairs (), [] and {} until nothing changes; valid if the string becomes empty.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Push every opener. On a closer, the top of the stack must be its matching opener; pop it. Valid if the stack ends empty.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Repeatedly delete adjacent pairs (), [] and {} until nothing changes; valid if the string becomes empty.

```python
def is_valid(s):
    prev = None
    while prev != s:
        prev = s
        s = s.replace("()", "").replace("[]", "").replace("{}", "")
    return s == ""
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Push every opener. On a closer, the top of the stack must be its matching opener; pop it. Valid if the stack ends empty.

```python
def is_valid(s):
    pair = {")": "(", "]": "[", "}": "{"}
    stack = []
    for i, ch in enumerate(s):
        if ch in pair:
            if not stack or stack.pop() != pair[ch]:
                return False
        else:
            stack.append(ch)
    return not stack
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeatedly delete adjacent pairs (), [] and {} until nothing changes; valid if the string becomes empty. That costs O(n²) time and O(n) space."

"I can do better: Push every opener. On a closer, the top of the stack must be its matching opener; pop it. Valid if the stack ends empty. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = '{[()]}(]'`

1. Push opener '{'
2. Push opener '['
3. Push opener '('
4. ')' must match the top '('
5. ']' must match the top '['
6. '}' must match the top '{'
7. Push opener '('
8. ']' must match the top '('

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/valid-parentheses)

<a id="q2"></a>
### Q2. Next Greater Element I

🟢 Warm-up · *Monotonic stack + map*

`nums1` is a subset of `nums2`. For each x in nums1, return the first greater number to the right of x in nums2, or −1.

**Examples**

- `nums1 = [4, 1, 2], nums2 = [1, 3, 4, 2]` → `[-1, 3, -1]`
- `nums1 = [2, 4], nums2 = [1, 2, 3, 4]` → `[3, -1]`

In [ ]:
def next_greater_element(nums1, nums2):
    """
    nums1 is a subset of nums2. For each x in nums1, return the first greater
    number to the right of x in nums2, or −1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(next_greater_element, [
    (([4, 1, 2], [1, 3, 4, 2]), [-1, 3, -1]),
    (([2, 4], [1, 2, 3, 4]), [3, -1]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each x, find it in nums2 and scan right.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One monotonic-stack pass over nums2 records next greater for every value in a dict; then look each x up.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each x, find it in nums2 and scan right.

```python
def next_greater_element(nums1, nums2):
    res = []
    for x in nums1:
        j = nums2.index(x)
        res.append(next((y for y in nums2[j + 1:] if y > x), -1))
    return res
```

**Cost:** O(n · m) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** One monotonic-stack pass over nums2 records next greater for every value in a dict; then look each x up.

```python
def next_greater_element(nums1, nums2):
    nxt, stack = {}, []
    for i, x in enumerate(nums2):
        while stack and stack[-1] < x:
            nxt[stack.pop()] = x
        stack.append(x)
    return [nxt.get(x, -1) for x in nums1]
```

**Cost:** O(n + m) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each x, find it in nums2 and scan right. That costs O(n · m) time and O(1) space."

"I can do better: One monotonic-stack pass over nums2 records next greater for every value in a dict; then look each x up. That's O(n + m) time and O(m) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums1 = [4, 1, 2], nums2 = [1, 3, 4, 2]`

1. After 1: known next-greater values
2. After 3: known next-greater values
3. After 4: known next-greater values
4. After 2: known next-greater values

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/next-greater-element-i)

<a id="q3"></a>
### Q3. Implement Queue using Stacks

🟢 Warm-up · *Two stacks · amortised*

Build a FIFO queue (`push`, `pop`, `peek`, `empty`) using only stack operations.

**Examples**

- `MyQueue() → push(1) → push(2) → peek() → pop() → empty()` → `[None, None, 1, 1, False]`

In [ ]:
class MyQueue:
    """
    Build a FIFO queue (push, pop, peek, empty) using only stack operations.
    """

    def __init__(self):
        pass

    def push(self, x):
        pass

    def peek(self):
        pass

    def pop(self):
        pass

    def empty(self):
        pass


# ▶ Run this cell to test your answer
check(MyQueue, [
    (((), [['push', (1,)], ['push', (2,)], ['peek', ()], ['pop', ()], ['empty', ()]]), [None, None, 1, 1, False]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

One stack; to pop the oldest, move everything to a temp stack and back.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Push onto 'in'. To pop or peek, if 'out' is empty pour all of 'in' into it — that reverses the order. Each element moves at most once.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** One stack; to pop the oldest, move everything to a temp stack and back.

```python
class MyQueue:
    def __init__(self):
        self.st = []

    def push(self, x):
        self.st.append(x)

    def pop(self):
        tmp = []
        while self.st:
            tmp.append(self.st.pop())
        x = tmp.pop()
        while tmp:
            self.st.append(tmp.pop())
        return x

    def peek(self):
        return self.st[0]

    def empty(self):
        return not self.st
```

**Cost:** pop O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Push onto 'in'. To pop or peek, if 'out' is empty pour all of 'in' into it — that reverses the order. Each element moves at most once.

```python
class MyQueue:
    def __init__(self):
        self.inp, self.out = [], []

    def push(self, x):
        self.inp.append(x)

    def _move(self):
        if not self.out:
            while self.inp:
                self.out.append(self.inp.pop())

    def pop(self):
        self._move()
        x = self.out.pop()
        return x

    def peek(self):
        self._move()
        return self.out[-1]

    def empty(self):
        return not self.inp and not self.out
```

**Cost:** O(1) amortised time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: One stack; to pop the oldest, move everything to a temp stack and back. That costs pop O(n) time and O(n) space."

"I can do better: Push onto 'in'. To pop or peek, if 'out' is empty pour all of 'in' into it — that reverses the order. Each element moves at most once. That's O(1) amortised time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `MyQueue() → push(1) → push(2) → push(3) → pop() → push(4) → pop() → peek()`

1. push(1) onto 'in'
2. push(2) onto 'in'
3. push(3) onto 'in'
4. 'out' was empty → pour 'in' into it (order reverses)
5. pop() → 1 from the top of 'out'
6. push(4) onto 'in'
7. pop() → 2 from the top of 'out'

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/implement-queue-using-stacks)

<a id="q4"></a>
### Q4. Implement Stack using Queues

🟢 Warm-up · *One queue · rotate*

Build a LIFO stack (`push`, `pop`, `top`, `empty`) using only queue operations.

**Examples**

- `MyStack() → push(1) → push(2) → top() → pop() → empty()` → `[None, None, 2, 2, False]`

In [ ]:
class MyStack:
    """
    Build a LIFO stack (push, pop, top, empty) using only queue operations.
    """

    def __init__(self):
        pass

    def push(self, x):
        pass

    def top(self):
        pass

    def pop(self):
        pass

    def empty(self):
        pass


# ▶ Run this cell to test your answer
check(MyStack, [
    (((), [['push', (1,)], ['push', (2,)], ['top', ()], ['pop', ()], ['empty', ()]]), [None, None, 2, 2, False]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two queues; on pop, move all but the last element to the other queue.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One queue. After appending x, rotate the queue len − 1 times so x comes to the front. Then pop and top are O(1).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Two queues; on pop, move all but the last element to the other queue.

```python
class MyStack:
    def __init__(self):
        from collections import deque
        self.q = deque()

    def push(self, x):
        self.q.append(x)

    def pop(self):
        from collections import deque
        other = deque()
        while len(self.q) > 1:
            other.append(self.q.popleft())
        x = self.q.popleft()
        self.q = other
        return x

    def top(self):
        return self.q[-1]

    def empty(self):
        return not self.q
```

**Cost:** pop O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** One queue. After appending x, rotate the queue len − 1 times so x comes to the front. Then pop and top are O(1).

```python
class MyStack:
    def __init__(self):
        from collections import deque
        self.q = deque()

    def push(self, x):
        self.q.append(x)
        for _ in range(len(self.q) - 1):
            self.q.append(self.q.popleft())

    def pop(self):
        x = self.q.popleft()
        return x

    def top(self):
        return self.q[0]

    def empty(self):
        return not self.q
```

**Cost:** push O(n), pop O(1) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two queues; on pop, move all but the last element to the other queue. That costs pop O(n) time and O(n) space."

"I can do better: One queue. After appending x, rotate the queue len − 1 times so x comes to the front. Then pop and top are O(1). That's push O(n), pop O(1) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `MyStack() → push(1) → push(2) → push(3) → pop() → top()`

1. push(1) then rotate so 1 is at the front
2. push(2) then rotate so 2 is at the front
3. push(3) then rotate so 3 is at the front
4. pop() → 3 from the front

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/implement-stack-using-queues)

<a id="q5"></a>
### Q5. Remove All Adjacent Duplicates In String

🟢 Warm-up · *Stack as the result*

Repeatedly remove two adjacent equal letters until none remain; return the result.

**Examples**

- `s = 'abbaca'` → `'ca'`
- `s = 'azxxzy'` → `'ay'`

In [ ]:
def remove_duplicates_str(s):
    """
    Repeatedly remove two adjacent equal letters until none remain; return the
    result.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(remove_duplicates_str, [
    (('abbaca',), 'ca'),
    (('azxxzy',), 'ay'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scan for an adjacent equal pair, delete it, and restart.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The stack holds the answer so far. If the next letter equals the top, pop (they cancel); otherwise push.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Scan for an adjacent equal pair, delete it, and restart.

```python
def remove_duplicates_str(s):
    changed = True
    while changed:
        changed = False
        for i in range(len(s) - 1):
            if s[i] == s[i + 1]:
                s = s[:i] + s[i + 2:]
                changed = True
                break
    return s
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** The stack holds the answer so far. If the next letter equals the top, pop (they cancel); otherwise push.

```python
def remove_duplicates_str(s):
    stack = []
    for i, ch in enumerate(s):
        if stack and stack[-1] == ch:
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scan for an adjacent equal pair, delete it, and restart. That costs O(n²) time and O(n) space."

"I can do better: The stack holds the answer so far. If the next letter equals the top, pop (they cancel); otherwise push. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'abbaca'`

1. Push 'a'
2. Push 'b'
3. 'b' equals the top → both cancel
4. 'a' equals the top → both cancel
5. Push 'c'
6. Push 'a'

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/remove-all-adjacent-duplicates-in-string)

<a id="q6"></a>
### Q6. Min Stack

🟡 Interview standard · *Stack of (value, min)*

Design a stack with `push`, `pop`, `top` and `getMin`, all in O(1).

**Examples**

- `MinStack() → push(-2) → push(0) → push(-3) → getMin() → pop() → top() → getMin()` → `[None, None, None, -3, None, 0, -2]`

In [ ]:
class MinStack:
    """
    Design a stack with push, pop, top and getMin, all in O(1).
    """

    def __init__(self):
        pass

    def push(self, val):
        pass

    def getMin(self):
        pass

    def pop(self):
        pass

    def top(self):
        pass


# ▶ Run this cell to test your answer
check(MinStack, [
    (((), [['push', (-2,)], ['push', (0,)], ['push', (-3,)], ['getMin', ()], ['pop', ()], ['top', ()], ['getMin', ()]]), [None, None, None, -3, None, 0, -2]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Plain list; getMin scans the whole stack.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Store each value with the minimum of the stack at that moment. The min is always at the top, and popping restores the previous min automatically.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Plain list; getMin scans the whole stack.

```python
class MinStack:
    def __init__(self):
        self.st = []

    def push(self, val):
        self.st.append(val)

    def pop(self):
        self.st.pop()

    def top(self):
        return self.st[-1]

    def getMin(self):
        return min(self.st)
```

**Cost:** getMin O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Store each value with the minimum of the stack at that moment. The min is always at the top, and popping restores the previous min automatically.

```python
class MinStack:
    def __init__(self):
        self.st = []  # (value, min so far)

    def push(self, val):
        low = min(val, self.st[-1][1]) if self.st else val
        self.st.append((val, low))

    def pop(self):
        self.st.pop()

    def top(self):
        return self.st[-1][0]

    def getMin(self):
        return self.st[-1][1]
```

**Cost:** O(1) every op time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Plain list; getMin scans the whole stack. That costs getMin O(n) time and O(n) space."

"I can do better: Store each value with the minimum of the stack at that moment. The min is always at the top, and popping restores the previous min automatically. That's O(1) every op time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `MinStack() → push(5) → push(3) → push(7) → push(2) → getMin() → pop() → getMin()`

1. push(5) → store (5, min 5)
2. push(3) → store (3, min 3)
3. push(7) → store (7, min 3)
4. push(2) → store (2, min 2)
5. getMin() = 2 straight from the top
6. pop() — the previous min is back on top
7. getMin() = 3 straight from the top

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/min-stack)

<a id="q7"></a>
### Q7. Evaluate Reverse Polish Notation

🟡 Interview standard · *Operand stack*

Evaluate an expression in Reverse Polish Notation (operators come after operands). Division truncates toward zero.

**Examples**

- `tokens = ['2', '1', '+', '3', '*']` → `9`
- `tokens = ['4', '13', '5', '/', '+']` → `6`
- `tokens = ['10', '6', '9', '3', '+', '-11', '*', '/', '*', '17', '+', '5', '+']` → `22`

In [ ]:
def eval_rpn(tokens):
    """
    Evaluate an expression in Reverse Polish Notation (operators come after
    operands). Division truncates toward zero.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(eval_rpn, [
    ((['2', '1', '+', '3', '*'],), 9),
    ((['4', '13', '5', '/', '+'],), 6),
    ((['10', '6', '9', '3', '+', '-11', '*', '/', '*', '17', '+', '5', '+'],), 22),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeatedly find the first operator, apply it to the two tokens before it, and splice the result back into the list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Push numbers. On an operator, pop b then a, push a op b. The last value is the answer.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Repeatedly find the first operator, apply it to the two tokens before it, and splice the result back into the list.

```python
def eval_rpn(tokens):
    tokens = list(tokens)
    while len(tokens) > 1:
        for i, t in enumerate(tokens):
            if t in "+-*/" and len(t) == 1:
                a, b = int(tokens[i - 2]), int(tokens[i - 1])
                v = a + b if t == "+" else a - b if t == "-" else a * b if t == "*" else int(a / b)
                tokens[i - 2:i + 1] = [str(v)]
                break
    return int(tokens[0])
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Push numbers. On an operator, pop b then a, push a op b. The last value is the answer.

```python
def eval_rpn(tokens):
    stack = []
    for i, t in enumerate(tokens):
        if t in ("+", "-", "*", "/"):
            b, a = stack.pop(), stack.pop()
            if t == "+":
                stack.append(a + b)
            elif t == "-":
                stack.append(a - b)
            elif t == "*":
                stack.append(a * b)
            else:
                stack.append(int(a / b))
        else:
            stack.append(int(t))
    return stack[0]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeatedly find the first operator, apply it to the two tokens before it, and splice the result back into the list. That costs O(n²) time and O(n) space."

"I can do better: Push numbers. On an operator, pop b then a, push a op b. The last value is the answer. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `tokens = ['2', '1', '+', '3', '*']`

1. Number 2 → push
2. Number 1 → push
3. '+': pop 1 and 2, push 2 + 1 = 3
4. Number 3 → push
5. '\*': pop 3 and 3, push 3 \* 3 = 9

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/evaluate-reverse-polish-notation)

<a id="q8"></a>
### Q8. Daily Temperatures

🟡 Interview standard · *Monotonic stack*

For each day, return how many days until a warmer temperature (0 if never).

**Examples**

- `temps = [73, 74, 75, 71, 69, 72, 76, 73]` → `[1, 1, 4, 2, 1, 1, 0, 0]`
- `temps = [30, 40, 50, 60]` → `[1, 1, 1, 0]`
- `temps = [30, 60, 90]` → `[1, 1, 0]`

In [ ]:
def daily_temperatures(temps):
    """
    For each day, return how many days until a warmer temperature (0 if never).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(daily_temperatures, [
    (([73, 74, 75, 71, 69, 72, 76, 73],), [1, 1, 4, 2, 1, 1, 0, 0]),
    (([30, 40, 50, 60],), [1, 1, 1, 0]),
    (([30, 60, 90],), [1, 1, 0]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each day, scan forward until a warmer day.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Stack of indices with decreasing temperatures. A warmer day pops every colder index and sets its wait to i − index.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each day, scan forward until a warmer day.

```python
def daily_temperatures(temps):
    res = [0] * len(temps)
    for i in range(len(temps)):
        for j in range(i + 1, len(temps)):
            if temps[j] > temps[i]:
                res[i] = j - i
                break
    return res
```

**Cost:** O(n²) time, O(1) extra space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Stack of indices with decreasing temperatures. A warmer day pops every colder index and sets its wait to i − index.

```python
def daily_temperatures(temps):
    res = [0] * len(temps)
    stack = []  # indices, temperatures decreasing
    for i, t in enumerate(temps):
        while stack and temps[stack[-1]] < t:
            j = stack.pop()
            res[j] = i - j
        stack.append(i)
    return res
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each day, scan forward until a warmer day. That costs O(n²) time and O(1) extra space."

"I can do better: Stack of indices with decreasing temperatures. A warmer day pops every colder index and sets its wait to i − index. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `temps = [73, 74, 75, 71, 69, 72, 76]`

1. 74 is warmer than day 0 (73) → wait 1
2. 75 is warmer than day 1 (74) → wait 1
3. 72 is warmer than day 4 (69) → wait 1
4. 72 is warmer than day 3 (71) → wait 2
5. 76 is warmer than day 5 (72) → wait 1
6. 76 is warmer than day 2 (75) → wait 4

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/daily-temperatures)

<a id="q9"></a>
### Q9. Next Greater Element II

🟡 Interview standard · *Monotonic stack · circular*

`nums` is circular. For every element return the next greater number going around, or −1.

**Examples**

- `nums = [1, 2, 1]` → `[2, -1, 2]`
- `nums = [1, 2, 3, 4, 3]` → `[2, 3, 4, -1, 4]`

In [ ]:
def next_greater_elements(nums):
    """
    nums is circular. For every element return the next greater number going
    around, or −1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(next_greater_elements, [
    (([1, 2, 1],), [2, -1, 2]),
    (([1, 2, 3, 4, 3],), [2, 3, 4, -1, 4]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each i, check the next n − 1 elements with wrap-around.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Run the monotonic stack over 2n steps using index i % n, so elements near the end can see the beginning.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each i, check the next n − 1 elements with wrap-around.

```python
def next_greater_elements(nums):
    n, res = len(nums), []
    for i in range(n):
        res.append(next((nums[(i + d) % n] for d in range(1, n) if nums[(i + d) % n] > nums[i]), -1))
    return res
```

**Cost:** O(n²) time, O(1) extra space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Run the monotonic stack over 2n steps using index i % n, so elements near the end can see the beginning.

```python
def next_greater_elements(nums):
    n = len(nums)
    res, stack = [-1] * n, []
    for k in range(2 * n):
        i = k % n
        while stack and nums[stack[-1]] < nums[i]:
            res[stack.pop()] = nums[i]
        if k < n:
            stack.append(i)
    return res
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each i, check the next n − 1 elements with wrap-around. That costs O(n²) time and O(1) extra space."

"I can do better: Run the monotonic stack over 2n steps using index i % n, so elements near the end can see the beginning. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 2, 1]`

1. Step 0 (index 0)
2. Step 1 (index 1)
3. Step 2 (index 2)
4. Step 3 (index 0, second lap)
5. Step 4 (index 1, second lap)
6. Step 5 (index 2, second lap)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/next-greater-element-ii)

<a id="q10"></a>
### Q10. Car Fleet

🟡 Interview standard · *Sort + stack of arrival times*

Cars drive toward `target`; a faster car that catches a slower one joins its fleet. Return the number of fleets that arrive.

**Examples**

- `target = 12, position = [10, 8, 0, 5, 3], speed = [2, 4, 1, 1, 3]` → `3`
- `target = 10, position = [3], speed = [3]` → `1`
- `target = 100, position = [0, 2, 4], speed = [4, 2, 1]` → `1`

In [ ]:
def car_fleet(target, position, speed):
    """
    Cars drive toward target; a faster car that catches a slower one joins its
    fleet. Return the number of fleets that arrive.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(car_fleet, [
    ((12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]), 3),
    ((10, [3], [3]), 1),
    ((100, [0, 2, 4], [4, 2, 1]), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate time step by step, merging cars that meet.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort by position, closest to target first. A car's arrival time ≤ the fleet ahead's time means it merges; otherwise it starts a new fleet.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Simulate time step by step, merging cars that meet.

```python
def car_fleet(target, position, speed):
    cars = sorted(zip(position, speed), reverse=True)
    fleets = 0
    times = [(target - p) / s for p, s in cars]
    for i in range(len(times)):
        if all(times[i] > times[j] for j in range(i)):
            fleets += 1
    return fleets
```

**Cost:** O(n · T) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort by position, closest to target first. A car's arrival time ≤ the fleet ahead's time means it merges; otherwise it starts a new fleet.

```python
def car_fleet(target, position, speed):
    cars = sorted(zip(position, speed), reverse=True)
    stack = []
    for p, s in cars:
        t = (target - p) / s
        if not stack or t > stack[-1]:
            stack.append(t)
    return len(stack)
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate time step by step, merging cars that meet. That costs O(n · T) time and O(n) space."

"I can do better: Sort by position, closest to target first. A car's arrival time ≤ the fleet ahead's time means it merges; otherwise it starts a new fleet. That's O(n log n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `target = 12, position = [10, 8, 0, 5, 3], speed = [2, 4, 1, 1, 3]`

1. Car at 10 arrives at t = 1.00; new fleet
2. Car at 8 arrives at t = 1.00; new fleet
3. Car at 5 arrives at t = 7.00; new fleet
4. Car at 3 arrives at t = 3.00; catches the fleet ahead
5. Car at 0 arrives at t = 12.00; new fleet

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/car-fleet)

<a id="q11"></a>
### Q11. Asteroid Collision

🟡 Interview standard · *Stack simulation*

Positive asteroids move right, negative move left. When two meet, the smaller explodes (both if equal). Return the survivors.

**Examples**

- `asteroids = [5, 10, -5]` → `[5, 10]`
- `asteroids = [8, -8]` → `[]`
- `asteroids = [10, 2, -5]` → `[10]`

In [ ]:
def asteroid_collision(asteroids):
    """
    Positive asteroids move right, negative move left. When two meet, the
    smaller explodes (both if equal). Return the survivors.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(asteroid_collision, [
    (([5, 10, -5],), [5, 10]),
    (([8, -8],), []),
    (([10, 2, -5],), [10]),
    (([-2, -1, 1, 2],), [-2, -1, 1, 2]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeatedly scan for a right-mover directly followed by a left-mover and resolve that collision.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Stack of survivors. A left-mover fights the right-movers on top of the stack until it dies or no right-mover remains.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Repeatedly scan for a right-mover directly followed by a left-mover and resolve that collision.

```python
def asteroid_collision(asteroids):
    a = list(asteroids)
    changed = True
    while changed:
        changed = False
        for i in range(len(a) - 1):
            if a[i] > 0 > a[i + 1]:
                if abs(a[i]) == abs(a[i + 1]):
                    del a[i:i + 2]
                elif abs(a[i]) > abs(a[i + 1]):
                    del a[i + 1]
                else:
                    del a[i]
                changed = True
                break
    return a
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Stack of survivors. A left-mover fights the right-movers on top of the stack until it dies or no right-mover remains.

```python
def asteroid_collision(asteroids):
    stack = []
    for i, x in enumerate(asteroids):
        alive = True
        while alive and x < 0 and stack and stack[-1] > 0:
            if stack[-1] < -x:
                stack.pop()
            elif stack[-1] == -x:
                stack.pop()
                alive = False
            else:
                alive = False
        if alive:
            stack.append(x)
    return stack
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeatedly scan for a right-mover directly followed by a left-mover and resolve that collision. That costs O(n²) time and O(n) space."

"I can do better: Stack of survivors. A left-mover fights the right-movers on top of the stack until it dies or no right-mover remains. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `asteroids = [5, 10, -5, -12, 3]`

1. After asteroid 5
2. After asteroid 10
3. -5 meets 10
4. After asteroid -5
5. -12 meets 10
6. -12 meets 5
7. After asteroid -12
8. After asteroid 3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/asteroid-collision)

<a id="q12"></a>
### Q12. Decode String

🟡 Interview standard · *Stack of (string, count)*

Decode strings like `3[a2[c]]` → `accaccacc`: `k[s]` means s repeated k times.

**Examples**

- `s = '3[a]2[bc]'` → `'aaabcbc'`
- `s = '3[a2[c]]'` → `'accaccacc'`
- `s = '2[abc]3[cd]ef'` → `'abcabccdcdcdef'`

In [ ]:
def decode_string(s):
    """
    Decode strings like 3[a2[c]] → accaccacc: k[s] means s repeated k times.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(decode_string, [
    (('3[a]2[bc]',), 'aaabcbc'),
    (('3[a2[c]]',), 'accaccacc'),
    (('2[abc]3[cd]ef',), 'abcabccdcdcdef'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeatedly find an innermost k[...] with a regex and expand it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Scan once. On '[' push (current string, k) and start fresh; on ']' pop and set cur = prev + cur × k.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Repeatedly find an innermost k[...] with a regex and expand it.

```python
def decode_string(s):
    import re
    pat = re.compile(r"(\d+)\[([a-z]*)\]")
    while "[" in s:
        s = pat.sub(lambda m: m.group(2) * int(m.group(1)), s)
    return s
```

**Cost:** O(n · output) time, O(output) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Scan once. On '[' push (current string, k) and start fresh; on ']' pop and set cur = prev + cur × k.

```python
def decode_string(s):
    stack, cur, k = [], "", 0
    for i, ch in enumerate(s):
        if ch.isdigit():
            k = k * 10 + int(ch)
        elif ch == "[":
            stack.append((cur, k))
            cur, k = "", 0
        elif ch == "]":
            prev, times = stack.pop()
            cur = prev + cur * times
        else:
            cur += ch
    return cur
```

**Cost:** O(output) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeatedly find an innermost k[...] with a regex and expand it. That costs O(n · output) time and O(output) space."

"I can do better: Scan once. On '[' push (current string, k) and start fresh; on ']' pop and set cur = prev + cur × k. That's O(output) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = '2[a3[b]]c'`

1. '[' → save (string so far, repeat count) and start fresh
2. '[' → save (string so far, repeat count) and start fresh
3. ']' → cur = prev + cur × 3
4. ']' → cur = prev + cur × 2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/decode-string)

<a id="q13"></a>
### Q13. Simplify Path

🟡 Interview standard · *Stack of directories*

Convert a Unix absolute path to its canonical form (resolve `.`, `..` and repeated slashes).

**Examples**

- `path = '/home/'` → `'/home'`
- `path = '/home//foo/'` → `'/home/foo'`
- `path = '/a/./b/../../c/'` → `'/c'`

In [ ]:
def simplify_path(path):
    """
    Convert a Unix absolute path to its canonical form (resolve ., .. and
    repeated slashes).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(simplify_path, [
    (('/home/',), '/home'),
    (('/home//foo/',), '/home/foo'),
    (('/a/./b/../../c/',), '/c'),
    (('/../',), '/'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeatedly apply string rewrites (remove '/./', collapse '//', remove 'dir/..') until stable.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Split on '/'. Push names, pop on '..', ignore '' and '.'. Join the stack.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Repeatedly apply string rewrites (remove '/./', collapse '//', remove 'dir/..') until stable.

```python
def simplify_path(path):
    import re
    p = path + "/"
    prev = None
    while prev != p:
        prev = p
        p = re.sub(r"/+", "/", p)
        p = p.replace("/./", "/")
        p = re.sub(r"/(?!\.\./)[^/]+/\.\./", "/", p, count=1)
        p = re.sub(r"^/\.\./", "/", p)
    return p.rstrip("/") or "/"
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Split on '/'. Push names, pop on '..', ignore '' and '.'. Join the stack.

```python
def simplify_path(path):
    stack = []
    parts = path.split("/")
    for i, part in enumerate(parts):
        if part == "..":
            if stack:
                stack.pop()
        elif part and part != ".":
            stack.append(part)
    return "/" + "/".join(stack)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeatedly apply string rewrites (remove '/./', collapse '//', remove 'dir/..') until stable. That costs O(n²) time and O(n) space."

"I can do better: Split on '/'. Push names, pop on '..', ignore '' and '.'. Join the stack. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `path = '/a/./b/../../c/d/'`

1. Part ''
2. Part 'a'
3. Part '.'
4. Part 'b'
5. Part '..'
6. Part '..'
7. Part 'c'
8. Part 'd'
9. Part ''

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/simplify-path)

<a id="q14"></a>
### Q14. Remove K Digits

🟡 Interview standard · *Monotonic stack · greedy*

Remove `k` digits from the number string `num` to make it as small as possible.

**Examples**

- `num = '1432219', k = 3` → `'1219'`
- `num = '10200', k = 1` → `'200'`
- `num = '10', k = 2` → `'0'`

In [ ]:
def remove_kdigits(num, k):
    """
    Remove k digits from the number string num to make it as small as possible.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(remove_kdigits, [
    (('1432219', 3), '1219'),
    (('10200', 1), '200'),
    (('10', 2), '0'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every way to remove k digits and keep the smallest.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A smaller digit should come as early as possible: while the stack top is bigger than the next digit and removals remain, pop it. Trim leftovers from the end and strip leading zeros.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every way to remove k digits and keep the smallest.

```python
def remove_kdigits(num, k):
    from itertools import combinations
    best = None
    for keep in combinations(range(len(num)), len(num) - k):
        s = "".join(num[i] for i in keep).lstrip("0") or "0"
        if best is None or (len(s), s) < (len(best), best):
            best = s
    return best or "0"
```

**Cost:** O(C(n, k) · n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** A smaller digit should come as early as possible: while the stack top is bigger than the next digit and removals remain, pop it. Trim leftovers from the end and strip leading zeros.

```python
def remove_kdigits(num, k):
    stack = []
    for i, d in enumerate(num):
        while k and stack and stack[-1] > d:
            stack.pop()
            k -= 1
        stack.append(d)
    stack = stack[:len(stack) - k] if k else stack
    return "".join(stack).lstrip("0") or "0"
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every way to remove k digits and keep the smallest. That costs O(C(n, k) · n) time and O(n) space."

"I can do better: A smaller digit should come as early as possible: while the stack top is bigger than the next digit and removals remain, pop it. Trim leftovers from the end and strip leading zeros. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `num = '1432219', k = 3`

1. Digit 1: pop bigger digits on top while removals remain
2. Digit 4: pop bigger digits on top while removals remain
3. Digit 3: pop bigger digits on top while removals remain
4. Digit 2: pop bigger digits on top while removals remain
5. Digit 2: pop bigger digits on top while removals remain
6. Digit 1: pop bigger digits on top while removals remain
7. Digit 9: pop bigger digits on top while removals remain

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/remove-k-digits)

<a id="q15"></a>
### Q15. Online Stock Span

🟡 Interview standard · *Monotonic stack · design*

For each new daily price, return the span: how many consecutive days (ending today) had price ≤ today's.

**Examples**

- `StockSpanner() → next(100) → next(80) → next(60) → next(70) → next(60) → next(75) → next(85)` → `[1, 1, 1, 2, 1, 4, 6]`

In [ ]:
class StockSpanner:
    """
    For each new daily price, return the span: how many consecutive days (ending
    today) had price ≤ today's.
    """

    def __init__(self):
        pass

    def next(self, price):
        pass


# ▶ Run this cell to test your answer
check(StockSpanner, [
    (((), [['next', (100,)], ['next', (80,)], ['next', (60,)], ['next', (70,)], ['next', (60,)], ['next', (75,)], ['next', (85,)]]), [1, 1, 1, 2, 1, 4, 6]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Store all prices; for each new price walk backwards while prices are ≤.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep a stack of (price, span) with decreasing prices. A new price absorbs every smaller-or-equal entry, adding their spans.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Store all prices; for each new price walk backwards while prices are ≤.

```python
class StockSpanner:
    def __init__(self):
        self.prices = []

    def next(self, price):
        self.prices.append(price)
        span = 0
        for p in reversed(self.prices):
            if p > price:
                break
            span += 1
        return span
```

**Cost:** O(n) per call time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Keep a stack of (price, span) with decreasing prices. A new price absorbs every smaller-or-equal entry, adding their spans.

```python
class StockSpanner:
    def __init__(self):
        self.stack = []  # (price, span)

    def next(self, price):
        span = 1
        while self.stack and self.stack[-1][0] <= price:
            span += self.stack.pop()[1]
        self.stack.append((price, span))
        return span
```

**Cost:** O(1) amortised time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Store all prices; for each new price walk backwards while prices are ≤. That costs O(n) per call time and O(n) space."

"I can do better: Keep a stack of (price, span) with decreasing prices. A new price absorbs every smaller-or-equal entry, adding their spans. That's O(1) amortised time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `StockSpanner() → next(100) → next(80) → next(60) → next(70) → next(60) → next(75) → next(85)`

1. next(100) → span 1
2. next(80) → span 1
3. next(60) → span 1
4. next(70) → span 2
5. next(60) → span 1
6. next(75) → span 4
7. next(85) → span 6

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/online-stock-span)

<a id="q16"></a>
### Q16. Sum of Subarray Minimums

🟡 Interview standard · *Monotonic stack · contribution*

Return the sum of min(sub) over every contiguous subarray, modulo 10⁹ + 7.

**Examples**

- `arr = [3, 1, 2, 4]` → `17`
- `arr = [11, 81, 94, 43, 3]` → `444`

In [ ]:
def sum_subarray_mins(arr):
    """
    Return the sum of min(sub) over every contiguous subarray, modulo 10⁹ + 7.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sum_subarray_mins, [
    (([3, 1, 2, 4],), 17),
    (([11, 81, 94, 43, 3],), 444),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Enumerate every subarray, tracking the running minimum.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Each arr[i] is the minimum of left[i] × right[i] subarrays, where left/right are distances to the previous smaller and next smaller-or-equal element. Find both with monotonic stacks.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Enumerate every subarray, tracking the running minimum.

```python
def sum_subarray_mins(arr):
    total = 0
    for i in range(len(arr)):
        low = arr[i]
        for j in range(i, len(arr)):
            low = min(low, arr[j])
            total += low
    return total % (10 ** 9 + 7)
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Each arr[i] is the minimum of left[i] × right[i] subarrays, where left/right are distances to the previous smaller and next smaller-or-equal element. Find both with monotonic stacks.

```python
def sum_subarray_mins(arr):
    n = len(arr)
    left, right, stack = [0] * n, [0] * n, []
    for i in range(n):
        while stack and arr[stack[-1]] > arr[i]:
            stack.pop()
        left[i] = i - stack[-1] if stack else i + 1
        stack.append(i)
    stack = []
    for i in range(n - 1, -1, -1):
        while stack and arr[stack[-1]] >= arr[i]:
            stack.pop()
        right[i] = stack[-1] - i if stack else n - i
        stack.append(i)
    total = 0
    for i in range(n):
        total += arr[i] * left[i] * right[i]
    return total % (10 ** 9 + 7)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Enumerate every subarray, tracking the running minimum. That costs O(n²) time and O(1) space."

"I can do better: Each arr[i] is the minimum of left[i] × right[i] subarrays, where left/right are distances to the previous smaller and next smaller-or-equal element. Find both with monotonic stacks. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `arr = [3, 1, 2, 4]`

1. 3 is the min of 1 × 1 = 1 subarrays
2. 1 is the min of 2 × 3 = 6 subarrays
3. 2 is the min of 1 × 2 = 2 subarrays
4. 4 is the min of 1 × 1 = 1 subarrays

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/sum-of-subarray-minimums)

<a id="q17"></a>
### Q17. Basic Calculator II

🟡 Interview standard · *Stack with deferred + and −*

Evaluate a string with non-negative integers, `+ - * /` and spaces. Division truncates toward zero.

**Examples**

- `s = '3+2*2'` → `7`
- `s = ' 3/2 '` → `1`
- `s = ' 3+5 / 2 '` → `5`

In [ ]:
def calculate(s):
    """
    Evaluate a string with non-negative integers, + - * / and spaces. Division
    truncates toward zero.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(calculate, [
    (('3+2*2',), 7),
    ((' 3/2 ',), 1),
    ((' 3+5 / 2 ',), 5),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Tokenise, then repeatedly resolve the first \* or / pair in the list, then the + and − left to right.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Read number by number. Push it for +, push −it for −, and for \* or / combine immediately with the stack top. The answer is the stack's sum.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Tokenise, then repeatedly resolve the first \* or / pair in the list, then the + and − left to right.

```python
def calculate(s):
    import re
    toks = re.findall(r"\d+|[+\-*/]", s)
    i = 1
    while i < len(toks):
        if toks[i] in "*/":
            a, b = int(toks[i - 1]), int(toks[i + 1])
            toks[i - 1:i + 2] = [str(a * b if toks[i] == "*" else int(a / b))]
        else:
            i += 2
    total = int(toks[0])
    for i in range(1, len(toks), 2):
        total += int(toks[i + 1]) if toks[i] == "+" else -int(toks[i + 1])
    return total
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Read number by number. Push it for +, push −it for −, and for \* or / combine immediately with the stack top. The answer is the stack's sum.

```python
def calculate(s):
    stack, num, op = [], 0, "+"
    for i, ch in enumerate(s + "+"):
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch in "+-*/":
            if op == "+":
                stack.append(num)
            elif op == "-":
                stack.append(-num)
            elif op == "*":
                stack.append(stack.pop() * num)
            else:
                stack.append(int(stack.pop() / num))
            op, num = ch, 0
    return sum(stack)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Tokenise, then repeatedly resolve the first \* or / pair in the list, then the + and − left to right. That costs O(n²) time and O(n) space."

"I can do better: Read number by number. Push it for +, push −it for −, and for \* or / combine immediately with the stack top. The answer is the stack's sum. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = '14-3*2+8/4'`

1. Apply pending '+' to 14
2. Apply pending '-' to 3
3. Apply pending '\*' to 2
4. Apply pending '+' to 8
5. Apply pending '/' to 4

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/basic-calculator-ii)

<a id="q18"></a>
### Q18. Largest Rectangle in Histogram

🔴 Stretch · *Monotonic stack · widths*

Given bar heights of width 1, return the area of the largest rectangle in the histogram.

**Examples**

- `heights = [2, 1, 5, 6, 2, 3]` → `10`
- `heights = [2, 4]` → `4`

In [ ]:
def largest_rectangle_area(heights):
    """
    Given bar heights of width 1, return the area of the largest rectangle in
    the histogram.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(largest_rectangle_area, [
    (([2, 1, 5, 6, 2, 3],), 10),
    (([2, 4],), 4),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each bar, expand left and right while bars are at least as tall.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep indices with increasing heights. When a shorter bar arrives, each popped bar's rectangle ends here and starts just after the new stack top — compute its area then.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each bar, expand left and right while bars are at least as tall.

```python
def largest_rectangle_area(heights):
    best = 0
    for i, h in enumerate(heights):
        l = r = i
        while l > 0 and heights[l - 1] >= h:
            l -= 1
        while r < len(heights) - 1 and heights[r + 1] >= h:
            r += 1
        best = max(best, h * (r - l + 1))
    return best
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Keep indices with increasing heights. When a shorter bar arrives, each popped bar's rectangle ends here and starts just after the new stack top — compute its area then.

```python
def largest_rectangle_area(heights):
    stack, best = [], 0  # indices, heights increasing
    for i in range(len(heights) + 1):
        h = heights[i] if i < len(heights) else 0
        while stack and heights[stack[-1]] >= h:
            top = stack.pop()
            left = stack[-1] + 1 if stack else 0
            area = heights[top] * (i - left)
            best = max(best, area)
        stack.append(i)
    return best
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each bar, expand left and right while bars are at least as tall. That costs O(n²) time and O(1) space."

"I can do better: Keep indices with increasing heights. When a shorter bar arrives, each popped bar's rectangle ends here and starts just after the new stack top — compute its area then. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `heights = [2, 1, 5, 6, 2, 3]`

1. Bar 2 spans 0..0: area 2
2. Bar 6 spans 3..3: area 6
3. Bar 5 spans 2..3: area 10
4. Bar 3 spans 5..5: area 3
5. Bar 2 spans 2..5: area 8
6. Bar 1 spans 0..5: area 6

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/largest-rectangle-in-histogram)

<a id="q19"></a>
### Q19. Maximal Rectangle

🔴 Stretch · *Histogram per row*

Given a binary matrix of '0' and '1', return the area of the largest rectangle containing only 1s.

**Examples**

- `matrix = [['1', '0', '1', '0', '0'], ['1', '0', '1', '1', '1'], ['1', '1', '1', '1', '1'], ['1', '0', '0', '1', '0']]` → `6`
- `matrix = [['0']]` → `0`
- `matrix = [['1']]` → `1`

In [ ]:
def maximal_rectangle(matrix):
    """
    Given a binary matrix of '0' and '1', return the area of the largest
    rectangle containing only 1s.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(maximal_rectangle, [
    (([['1', '0', '1', '0', '0'], ['1', '0', '1', '1', '1'], ['1', '1', '1', '1', '1'], ['1', '0', '0', '1', '0']],), 6),
    (([['0']],), 0),
    (([['1']],), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every top-left and bottom-right corner and check the rectangle is all 1s.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Treat each row as the base of a histogram: heights[c] = consecutive 1s ending at this row. Run Largest Rectangle in Histogram on every row.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every top-left and bottom-right corner and check the rectangle is all 1s.

```python
def maximal_rectangle(matrix):
    m, n, best = len(matrix), len(matrix[0]), 0
    for r1 in range(m):
        for c1 in range(n):
            for r2 in range(r1, m):
                for c2 in range(c1, n):
                    if all(matrix[r][c] == "1" for r in range(r1, r2 + 1) for c in range(c1, c2 + 1)):
                        best = max(best, (r2 - r1 + 1) * (c2 - c1 + 1))
    return best
```

**Cost:** O((mn)³) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Treat each row as the base of a histogram: heights[c] = consecutive 1s ending at this row. Run Largest Rectangle in Histogram on every row.

```python
def maximal_rectangle(matrix):
    n = len(matrix[0])
    heights, best = [0] * n, 0
    for r, row in enumerate(matrix):
        for c in range(n):
            heights[c] = heights[c] + 1 if row[c] == "1" else 0
        stack = []
        for i in range(n + 1):
            h = heights[i] if i < n else 0
            while stack and heights[stack[-1]] >= h:
                top = stack.pop()
                left = stack[-1] + 1 if stack else 0
                best = max(best, heights[top] * (i - left))
            stack.append(i)
    return best
```

**Cost:** O(m · n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every top-left and bottom-right corner and check the rectangle is all 1s. That costs O((mn)³) time and O(1) space."

"I can do better: Treat each row as the base of a histogram: heights[c] = consecutive 1s ending at this row. Run Largest Rectangle in Histogram on every row. That's O(m · n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `matrix = [['1', '0', '1', '0', '0'], ['1', '0', '1', '1', '1'], ['1', '1', '1', '1', '1'], ['1', '0', '0', '1', '0']]`

1. Row 0 as the floor → histogram [1, 0, 1, 0, 0]; best so far 1
2. Row 1 as the floor → histogram [2, 0, 2, 1, 1]; best so far 3
3. Row 2 as the floor → histogram [3, 1, 3, 2, 2]; best so far 6
4. Row 3 as the floor → histogram [4, 0, 0, 3, 0]; best so far 6

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/maximal-rectangle)

<a id="q20"></a>
### Q20. Longest Valid Parentheses

🔴 Stretch · *Stack of indices*

Return the length of the longest well-formed parentheses substring.

**Examples**

- `s = '(()'` → `2`
- `s = ')()())'` → `4`
- `s = ''` → `0`

In [ ]:
def longest_valid_parentheses(s):
    """
    Return the length of the longest well-formed parentheses substring.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(longest_valid_parentheses, [
    (('(()',), 2),
    ((')()())',), 4),
    (('',), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check every even-length substring for validity.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Stack of indices, seeded with −1 as the base. '(' pushes its index. ')' pops; if the stack is empty, this index becomes the new base, otherwise i − stack top is a valid length.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Check every even-length substring for validity.

```python
def longest_valid_parentheses(s):
    def ok(t):
        bal = 0
        for c in t:
            bal += 1 if c == "(" else -1
            if bal < 0:
                return False
        return bal == 0
    best = 0
    for i in range(len(s)):
        for j in range(i + 2, len(s) + 1, 2):
            if ok(s[i:j]):
                best = max(best, j - i)
    return best
```

**Cost:** O(n³) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Stack of indices, seeded with −1 as the base. '(' pushes its index. ')' pops; if the stack is empty, this index becomes the new base, otherwise i − stack top is a valid length.

```python
def longest_valid_parentheses(s):
    stack, best = [-1], 0
    for i, ch in enumerate(s):
        if ch == "(":
            stack.append(i)
        else:
            stack.pop()
            if not stack:
                stack.append(i)
            else:
                best = max(best, i - stack[-1])
    return best
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check every even-length substring for validity. That costs O(n³) time and O(1) space."

"I can do better: Stack of indices, seeded with −1 as the base. '(' pushes its index. ')' pops; if the stack is empty, this index becomes the new base, otherwise i − stack top is a valid length. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = '()(())('`

1. '(' at 0: valid run ends here of length 0
2. ')' at 1: valid run ends here of length 2
3. '(' at 2: valid run ends here of length 0
4. '(' at 3: valid run ends here of length 0
5. ')' at 4: valid run ends here of length 2
6. ')' at 5: valid run ends here of length 6
7. '(' at 6: valid run ends here of length 0

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/stack-queue/longest-valid-parentheses)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Stack vs queue — what's the difference?</b></summary>

A stack removes the most recently added element (LIFO); a queue removes the oldest (FIFO). Stack: one open end. Queue: insert at one end, remove at the other.

</details>

<details>
<summary><b>2. Why not use a Python list as a queue?</b></summary>

`list.pop(0)` shifts every remaining element: O(n). `collections.deque.popleft()` is O(1) because deque is a doubly-linked block structure.

</details>

<details>
<summary><b>3. What is a circular queue and why use it?</b></summary>

A fixed-size array queue where indices wrap around with modulo. It reuses slots freed at the front, so a simple array queue doesn't run out of space while elements are still free at the start.

</details>

<details>
<summary><b>4. How do you implement a queue using two stacks? What's the complexity?</b></summary>

Push to stack `in`. For pop/peek, if `out` is empty, move all of `in` into `out` (reversing the order), then pop from `out`. Each element is moved at most once, so operations are amortised O(1).

</details>

<details>
<summary><b>5. What is a monotonic stack?</b></summary>

A stack kept in increasing or decreasing order by popping elements that would break the order before pushing. Each pop answers a 'next greater/smaller' question for the popped element. O(n) total.

</details>

<details>
<summary><b>6. How is recursion related to the stack?</b></summary>

Each function call pushes a frame (arguments, local variables, return address) onto the call stack; returning pops it. Any recursive algorithm can be rewritten with an explicit stack, which also avoids recursion-depth limits.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: The stack: last in, first out
- [ ] I can explain: The queue: first in, first out
- [ ] I can explain: Matching and nesting
- [ ] I can explain: Evaluating expressions
- [ ] I can explain: Monotonic stack: next greater element
- [ ] I can explain: Monotonic stack for ranges and contributions
- [ ] I can explain: Designing with stacks and queues

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Sum of Subarray Minimums
- [ ] [Q17](#q17) Basic Calculator II
- [ ] [Q18](#q18) Largest Rectangle in Histogram
- [ ] [Q19](#q19) Maximal Rectangle
- [ ] [Q20](#q20) Longest Valid Parentheses

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.